# Pure-State Dynamic Square Correlations

This local CPU notebook reads the saved pure-state covariance snapshots and computes the fixed-`x` squared correlation observable along the periodic `y` direction. It does not run new adaptive-circuit dynamics; it validates that the source snapshots were produced by `classA_U1FGTN_gpu.run_markov_circuit(...)` and treats the covariance files as read-only inputs.

The notebook is organized in two scopes: first, one time-resolved correlation-profile section with one plotting cell per system size; second, cycle-50-only log-chord and fixed-`x` slope diagnostics.

Source campaign:
`colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/campaign_manifest.json`

Outputs:
`colab_small_system_testing/analysis_outputs/pure_state_square_correlations_cpu/`


## Configurations And Observable

The source campaign contains four saved snapshot runs:

| Nx | Ny | nshell | protocol | samples | cycles | saved snapshots | init state | dtype | dw truncation |
|---:|---:|---:|---|---:|---:|---|---|---|---|
| 16 | 30 | 1 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default pure | complex128 | True |
| 16 | 30 | 2 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default pure | complex128 | True |
| 16 | 40 | 1 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default pure | complex128 | True |
| 16 | 40 | 2 | perfect_correction | 10 | 50 | 5, 10, 20, 50 | default pure | complex128 | True |

For each top-layer covariance snapshot `G`, the notebook forms `C = (G + I)/2` and computes

```text
C_G(x, ry) = (1 / (2 Ny)) sum_y sum_{mu,nu=0,1} |C[x,y,mu; x,y+ry,nu]|^2,
```

with `y + ry` interpreted modulo `Ny`. Fits use `log <C_G(ry)>_x` versus `log chord(ry)` over `ry=3..10`, where `chord(ry) = (Ny/pi) sin(pi ry / Ny)`.


## 1. Setup

Locate the local bundle and project source tree, configure output paths, and import the exact equilibrium reference class.


In [1]:
from __future__ import annotations

import contextlib
import io
import json
import math
import os
import sys
import warnings
from pathlib import Path

Path('/tmp/matplotlib-cache').mkdir(parents=True, exist_ok=True)
Path('/tmp/xdg-cache').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/xdg-cache')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BUNDLE_NAME = 'colab_small_system_testing'
CANONICAL_SOURCE_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
SOURCE_CAMPAIGN = 'pure_state_covariance_snapshots'
ANALYSIS_NAME = 'pure_state_square_correlations_cpu'

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / 'gpu_data' / SOURCE_CAMPAIGN / 'campaign_manifest.json').exists():
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME}/gpu_data/{SOURCE_CAMPAIGN}')

PROJECT_ROOT = BUNDLE_ROOT.parent
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from fgtn.classA_U1FGTN import classA_U1FGTN

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
SOURCE_ROOT = GPU_DATA_ROOT / SOURCE_CAMPAIGN
SOURCE_MANIFEST_PATH = SOURCE_ROOT / 'campaign_manifest.json'
OUTPUT_ROOT = BUNDLE_ROOT / 'analysis_outputs' / ANALYSIS_NAME
FIG_ROOT = OUTPUT_ROOT / 'figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)
for stale_figure in FIG_ROOT.glob('*.png'):
    stale_figure.unlink()

EXPECTED_CONFIGS = {(16, 30, 1), (16, 30, 2), (16, 40, 1), (16, 40, 2)}
EXPECTED_SNAPSHOT_CYCLES = np.asarray([5, 10, 20, 50], dtype=np.int64)
EXPECTED_SHARD_SHAPES = {
    (16, 30): (10, 4, 960, 960),
    (16, 40): (10, 4, 1280, 1280),
}
PRIMARY_CYCLE = 50
CORR_FIT_RY_MIN = 3
CORR_FIT_RY_MAX = 10
NEGATIVE_TOL = 1e-14

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] project root: {PROJECT_ROOT}')
print(f'[setup] source manifest: {SOURCE_MANIFEST_PATH}')
print(f'[setup] output root: {OUTPUT_ROOT}')


[setup] bundle root: /home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing
[setup] project root: /home/abhuiyan/class_A_fermionic_adaptive_circuit
[setup] source manifest: /home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/campaign_manifest.json
[setup] output root: /home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/analysis_outputs/pure_state_square_correlations_cpu


## 2. Shared Helpers

Utilities for JSON/NPZ output, source path normalization, the square-correlation observable, exact references, and log-fit diagnostics.


In [2]:
def load_json(path: Path):
    with Path(path).open('r', encoding='utf-8') as fh:
        return json.load(fh)


def write_json_atomic(path: Path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp.replace(path)


def save_npz_atomic(path: Path, **arrays) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as fh:
        np.savez_compressed(fh, **arrays)
    tmp.replace(path)


def rel_to_bundle(path: Path) -> str:
    return str(Path(path).resolve().relative_to(BUNDLE_ROOT.resolve()))


def rel_to_gpu_data(path: Path) -> str:
    return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))


def chord_length(ry, circumference: int):
    ry = np.asarray(ry, dtype=np.float64)
    return (float(circumference) / np.pi) * np.sin(np.pi * ry / float(circumference))


def fit_linear(x, y, min_points: int = 3) -> dict:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < min_points:
        return {'slope': np.nan, 'intercept': np.nan, 'r2': np.nan, 'n_fit_points': int(x.size)}
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    return {
        'slope': float(slope),
        'intercept': float(intercept),
        'r2': float(r2),
        'n_fit_points': int(x.size),
    }


def default_domain_wall_locations(nx: int) -> tuple[int, int]:
    half = int(nx) // 2
    width = max(1, int(np.floor(0.2 * int(nx))))
    x0 = max(0, half - width)
    x1_exclusive = min(int(nx), half + width + 1)
    return int(x0), int(x1_exclusive - 1)


def top_layer_covariance(g, nx: int, ny: int) -> np.ndarray:
    nx = int(nx)
    ny = int(ny)
    nlayer = 2 * nx * ny
    arr = np.asarray(g)
    if arr.shape == (2 * nlayer, 2 * nlayer):
        return np.asarray(arr[:nlayer, :nlayer])
    if arr.shape == (nlayer, nlayer):
        return np.asarray(arr)
    raise ValueError(f'Expected top-layer ({nlayer},{nlayer}) or full ({2*nlayer},{2*nlayer}) covariance; got {arr.shape}')


def square_corr_from_G_top(G_top, nx: int, ny: int) -> tuple[np.ndarray, np.ndarray]:
    nx = int(nx)
    ny = int(ny)
    nlayer = 2 * nx * ny
    g_top = top_layer_covariance(G_top, nx, ny)
    c = 0.5 * (g_top.astype(np.complex128, copy=False) + np.eye(nlayer, dtype=np.complex128))
    c6 = c.reshape(2, nx, ny, 2, nx, ny, order='F')
    gker = np.transpose(c6, (1, 2, 0, 4, 5, 3))
    ry_values = np.arange(0, ny // 2 + 1, dtype=np.int64)
    corr = np.empty((nx, ry_values.size), dtype=np.float64)
    y = np.arange(ny, dtype=np.int64)[:, None]
    yp = (y + ry_values[None, :]) % ny
    flat_ix = (y * ny + yp).reshape(-1)
    for x0 in range(nx):
        gx = gker[x0, :, :, x0, :, :]
        gx_re = np.transpose(gx, (0, 2, 1, 3)).reshape(ny * ny, 2, 2)
        blocks = gx_re[flat_ix].reshape(ny, ry_values.size, 2, 2)
        corr[x0] = np.sum(np.abs(blocks) ** 2, axis=(0, 2, 3)) / (2.0 * ny)
    return ry_values, corr


def square_corr_slow_subset(G_top, nx: int, ny: int, x_values, ry_values) -> dict[tuple[int, int], float]:
    nx = int(nx)
    ny = int(ny)
    nlayer = 2 * nx * ny
    g_top = top_layer_covariance(G_top, nx, ny)
    c = 0.5 * (g_top.astype(np.complex128, copy=False) + np.eye(nlayer, dtype=np.complex128))
    out = {}
    for x0 in [int(x) for x in x_values]:
        for ry in [int(r) for r in ry_values]:
            total = 0.0
            for y in range(ny):
                yp = (y + ry) % ny
                for mu in range(2):
                    i = mu + 2 * x0 + 2 * nx * y
                    for nu in range(2):
                        j = nu + 2 * x0 + 2 * nx * yp
                        total += abs(c[i, j]) ** 2
            out[(x0, ry)] = float(total / (2.0 * ny))
    return out


def validate_square_corr_spotcheck(G_top, nx: int, ny: int) -> dict:
    ry_values, fast = square_corr_from_G_top(G_top, nx, ny)
    dw0, dw1 = default_domain_wall_locations(nx)
    x_subset = sorted({0, max(0, dw0 - 1), dw0, min(nx - 1, dw0 + 1), dw1, min(nx - 1, dw1 + 1), nx - 1})
    ry_subset = [0, 1, min(3, ny // 2), min(10, ny // 2)]
    slow = square_corr_slow_subset(G_top, nx, ny, x_subset, ry_subset)
    max_abs_diff = 0.0
    for (x0, ry), value in slow.items():
        r_idx = int(np.flatnonzero(ry_values == ry)[0])
        max_abs_diff = max(max_abs_diff, abs(float(fast[x0, r_idx]) - value))
    if max_abs_diff > 1e-11:
        raise AssertionError(f'square-correlation spot check failed: max_abs_diff={max_abs_diff:g}')
    return {
        'x_values': x_subset,
        'ry_values': ry_subset,
        'max_abs_diff': float(max_abs_diff),
    }


def selected_x_values(nx: int) -> list[int]:
    dw0, dw1 = default_domain_wall_locations(nx)
    values = {0, max(0, dw0 - 1), dw0, min(nx - 1, dw0 + 1), nx // 2, max(0, dw1 - 1), dw1, min(nx - 1, dw1 + 1), nx - 1}
    return sorted(values)


def fit_corr_curve(ry_values, corr_values, ny: int) -> dict:
    ry_values = np.asarray(ry_values, dtype=np.float64)
    corr_values = np.asarray(corr_values, dtype=np.float64)
    chord = chord_length(ry_values, int(ny))
    valid = (
        (ry_values > 0)
        & (ry_values < float(ny) / 2.0)
        & (ry_values >= CORR_FIT_RY_MIN)
        & (ry_values <= CORR_FIT_RY_MAX)
        & (chord > 0)
        & (corr_values > 0)
        & np.isfinite(corr_values)
    )
    fit = fit_linear(np.log(chord[valid]), np.log(corr_values[valid]))
    fit.update({
        'ry_fit_min': int(np.min(ry_values[valid])) if np.any(valid) else np.nan,
        'ry_fit_max': int(np.max(ry_values[valid])) if np.any(valid) else np.nan,
        'fit_coordinate': 'log_chord_ry',
    })
    return fit


def exact_square_corr_reference(nx: int, ny: int) -> dict:
    with contextlib.redirect_stdout(io.StringIO()):
        model = classA_U1FGTN(int(nx), int(ny), DW=True, alpha_1=1.0, alpha_2=30.0)
        g_exact = model.G_CI_domain_wall(periodic=True)
    ry_values, corr = square_corr_from_G_top(g_exact, nx, ny)
    dw = tuple(int(v) for v in getattr(model, 'DW_loc', default_domain_wall_locations(nx)))
    return {
        'Nx': int(nx),
        'Ny': int(ny),
        'domain_wall_x': dw,
        'ry_values': ry_values,
        'corr_sq': corr,
        'xavg_corr': np.mean(corr, axis=0),
    }


def show_saved_figure(path: Path, title: str | None = None, note: str | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if title:
        display(Markdown(f'**{title}**'))
    if note:
        display(Markdown(note))
    display(Image(filename=str(path)))


## 3. Load And Validate Source Campaign

Validate the source campaign and construct one record per covariance snapshot shard. The notebook fails before computation if any expected campaign invariant is violated.


In [3]:
def validate_source_entry(entry: dict, manifest: dict, summary: dict, shard_path: Path) -> dict:
    nx, ny, nshell = int(entry['Nx']), int(entry['Ny']), int(entry['nshell'])
    key = (nx, ny, nshell)
    if key not in EXPECTED_CONFIGS:
        raise ValueError(f'Unexpected source config {key}')
    if int(entry.get('samples', -1)) != 10 or int(entry.get('cycles', -1)) != 50:
        raise ValueError(f'{key}: expected samples=10 and cycles=50')
    if [int(c) for c in entry.get('snapshot_cycles', [])] != EXPECTED_SNAPSHOT_CYCLES.tolist():
        raise ValueError(f'{key}: unexpected entry snapshot cycles {entry.get("snapshot_cycles")}')
    if entry.get('protocol') != 'perfect_correction' or entry.get('init_mode') != 'default':
        raise ValueError(f'{key}: unexpected protocol/init {entry.get("protocol")}/{entry.get("init_mode")}')
    if entry.get('dtype_resolved') != 'complex128':
        raise ValueError(f'{key}: unexpected entry dtype {entry.get("dtype_resolved")}')
    if entry.get('canonical_dynamics_entry_point') != CANONICAL_SOURCE_ENTRY_POINT:
        raise ValueError(f'{key}: unexpected canonical source entry point in campaign entry')

    if summary.get('canonical_dynamics_entry_point') != CANONICAL_SOURCE_ENTRY_POINT:
        raise ValueError(f'{key}: unexpected source entry point in run_summary.json')
    if summary.get('protocol') != 'perfect_correction' or summary.get('init_mode') != 'default':
        raise ValueError(f'{key}: unexpected summary protocol/init')
    if summary.get('dtype_resolved') != 'complex128':
        raise ValueError(f'{key}: unexpected summary dtype')

    cfg = manifest.get('config', {})
    if manifest.get('store_mode') != 'snapshots' or cfg.get('store_mode') != 'snapshots':
        raise ValueError(f'{key}: expected snapshot store mode')
    if manifest.get('num_batches') != 1:
        raise ValueError(f'{key}: expected one shard, got {manifest.get("num_batches")}')
    if cfg.get('dtype') != 'c128':
        raise ValueError(f'{key}: expected manifest dtype c128, got {cfg.get("dtype")}')
    if cfg.get('dw_truncation') is not True:
        raise ValueError(f'{key}: expected dw_truncation=True')
    if cfg.get('trial_orbitals') != 'X':
        raise ValueError(f'{key}: expected trial_orbitals=X')
    if cfg.get('postselect') is not False or cfg.get('perfect_correction') is not True:
        raise ValueError(f'{key}: expected non-postselected perfect correction')
    if [int(c) for c in cfg.get('snapshot_cycles', [])] != EXPECTED_SNAPSHOT_CYCLES.tolist():
        raise ValueError(f'{key}: unexpected manifest snapshot cycles')

    shards = manifest.get('shards', {})
    if sorted(int(k) for k in shards) != [0]:
        raise ValueError(f'{key}: expected exactly shard 0, got {sorted(shards)}')
    expected_shape = EXPECTED_SHARD_SHAPES[(nx, ny)]
    shard_meta = shards['0']
    if [int(x) for x in shard_meta.get('shape', [])] != list(expected_shape):
        raise ValueError(f'{key}: manifest shard shape mismatch {shard_meta.get("shape")} vs {expected_shape}')
    if not shard_path.exists():
        raise FileNotFoundError(shard_path)
    shard = np.load(shard_path, mmap_mode='r')
    if shard.shape != expected_shape or shard.dtype != np.complex128:
        raise ValueError(f'{shard_path}: expected {expected_shape} complex128, got {shard.shape} {shard.dtype}')
    if int(shard_meta.get('sample_start', 0)) != 0 or int(shard_meta.get('sample_stop', -1)) != 10:
        raise ValueError(f'{key}: unexpected sample range {shard_meta}')
    return {
        'Nx': nx,
        'Ny': ny,
        'nshell': nshell,
        'expected_shape': expected_shape,
        'domain_wall_x': default_domain_wall_locations(nx),
    }


def source_run_records() -> list[dict]:
    campaign = load_json(SOURCE_MANIFEST_PATH)
    if campaign.get('canonical_dynamics_entry_point') != CANONICAL_SOURCE_ENTRY_POINT:
        raise ValueError(f'Unexpected campaign entry point in {SOURCE_MANIFEST_PATH}')
    if campaign.get('protocol') != 'perfect_correction' or campaign.get('dtype') != 'complex128':
        raise ValueError(f'Unexpected source campaign protocol/dtype in {SOURCE_MANIFEST_PATH}')
    if [int(c) for c in campaign.get('snapshot_cycles', [])] != EXPECTED_SNAPSHOT_CYCLES.tolist():
        raise ValueError(f'Unexpected campaign snapshot cycles {campaign.get("snapshot_cycles")}')
    results = campaign.get('results', [])
    if len(results) != len(EXPECTED_CONFIGS):
        raise ValueError(f'Expected {len(EXPECTED_CONFIGS)} source configs, found {len(results)}')

    records = []
    seen = set()
    for entry in results:
        run_dir = GPU_DATA_ROOT / entry['run_dir_relative']
        manifest_path = GPU_DATA_ROOT / entry['manifest_path_relative']
        summary_path = GPU_DATA_ROOT / entry['summary_path_relative']
        manifest = load_json(manifest_path)
        summary = load_json(summary_path)
        shard_path = run_dir / manifest['shards']['0']['filename']
        validated = validate_source_entry(entry, manifest, summary, shard_path)
        key = (validated['Nx'], validated['Ny'], validated['nshell'])
        if key in seen:
            raise ValueError(f'Duplicate source config {key}')
        seen.add(key)
        case_id = f"N{validated['Nx']}x{validated['Ny']}_nsh{validated['nshell']}_{entry['protocol']}"
        records.append({
            **entry,
            **validated,
            'case_id': case_id,
            'run_dir': run_dir,
            'manifest_path': manifest_path,
            'summary_path': summary_path,
            'shard_path': shard_path,
            'source_manifest': manifest,
            'source_summary': summary,
        })

    if seen != EXPECTED_CONFIGS:
        raise ValueError(f'Source config mismatch: observed={sorted(seen)} expected={sorted(EXPECTED_CONFIGS)}')
    return sorted(records, key=lambda rec: (int(rec['Ny']), int(rec['nshell'])))


SOURCE_RUNS = source_run_records()
validation_report = {
    'source_manifest': rel_to_bundle(SOURCE_MANIFEST_PATH),
    'canonical_source_entry_point': CANONICAL_SOURCE_ENTRY_POINT,
    'expected_snapshot_cycles': EXPECTED_SNAPSHOT_CYCLES.astype(int).tolist(),
    'runs': {},
}
for record in SOURCE_RUNS:
    print(f"[source] {record['case_id']} shard={record['shard_path']} shape={record['expected_shape']}")


[source] N16x30_nsh1_perfect_correction shard=/home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/runs/N16x30_nsh1_perfect_correction/run_7e11a0a6345c/batch_00000_snapshots.npy shape=(10, 4, 960, 960)
[source] N16x30_nsh2_perfect_correction shard=/home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/runs/N16x30_nsh2_perfect_correction/run_60d11c578e5b/batch_00000_snapshots.npy shape=(10, 4, 960, 960)
[source] N16x40_nsh1_perfect_correction shard=/home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/runs/N16x40_nsh1_perfect_correction/run_3301a140f933/batch_00000_snapshots.npy shape=(10, 4, 1280, 1280)
[source] N16x40_nsh2_perfect_correction shard=/home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/gpu_data/pure_state_covariance_snapshots/runs/N16x40_nsh2_perfect_correcti

## 4. Compute Dynamic Square Correlations

Compute sample-averaged fixed-`x` square correlations for every saved cycle. The first source snapshot is also checked against a direct slow-loop implementation for selected `(x, ry)` entries.


In [4]:
def compute_record_square_correlations(record: dict) -> dict:
    nx, ny = int(record['Nx']), int(record['Ny'])
    snapshot_cycles = EXPECTED_SNAPSHOT_CYCLES.astype(int)
    shard = np.load(record['shard_path'], mmap_mode='r')
    if shard.shape != tuple(record['expected_shape']):
        raise ValueError(f"{record['case_id']}: shape changed while loading: {shard.shape}")

    spotcheck = validate_square_corr_spotcheck(shard[0, 0], nx, ny)
    print(f"[spotcheck] {record['case_id']} max_abs_diff={spotcheck['max_abs_diff']:.3e}")

    sample_count = int(shard.shape[0])
    time_count = int(shard.shape[1])
    ry_values = np.arange(0, ny // 2 + 1, dtype=np.int64)
    corr_sum = np.zeros((time_count, nx, ry_values.size), dtype=np.float64)
    corr_sumsq = np.zeros_like(corr_sum)

    for s_idx in range(sample_count):
        for t_idx in range(time_count):
            ry_check, corr = square_corr_from_G_top(shard[s_idx, t_idx], nx, ny)
            if not np.array_equal(ry_check, ry_values):
                raise ValueError(f"{record['case_id']}: ry grid mismatch")
            if not np.all(np.isfinite(corr)):
                raise FloatingPointError(f"{record['case_id']}: non-finite correlation values at sample={s_idx}, t={t_idx}")
            min_corr = float(np.min(corr))
            if min_corr < -NEGATIVE_TOL:
                raise ValueError(f"{record['case_id']}: negative correlation value {min_corr:g}")
            corr_sum[t_idx] += corr
            corr_sumsq[t_idx] += corr ** 2

    corr_mean = corr_sum / float(sample_count)
    if sample_count > 1:
        variance = (corr_sumsq - (corr_sum ** 2) / float(sample_count)) / float(sample_count - 1)
        variance = np.maximum(variance, 0.0)
        corr_std = np.sqrt(variance)
        corr_sem = corr_std / math.sqrt(float(sample_count))
    else:
        corr_std = np.zeros_like(corr_mean)
        corr_sem = np.zeros_like(corr_mean)

    xavg_mean = np.mean(corr_mean, axis=1)
    xavg_sem = np.sqrt(np.sum(corr_sem ** 2, axis=1)) / float(nx)

    if not np.all(np.isfinite(corr_mean)) or not np.all(np.isfinite(xavg_mean)):
        raise FloatingPointError(f"{record['case_id']}: non-finite averaged output")
    if float(np.min(corr_mean)) < -NEGATIVE_TOL:
        raise ValueError(f"{record['case_id']}: negative averaged correlation")

    validation_report['runs'][record['case_id']] = {
        'Nx': nx,
        'Ny': ny,
        'nshell': int(record['nshell']),
        'shard_path': rel_to_bundle(record['shard_path']),
        'expected_shape': list(record['expected_shape']),
        'sample_count': sample_count,
        'snapshot_cycles': snapshot_cycles.tolist(),
        'domain_wall_x': list(record['domain_wall_x']),
        'spotcheck': spotcheck,
        'corr_min': float(np.min(corr_mean)),
        'corr_max': float(np.max(corr_mean)),
    }

    return {
        'record': record,
        'snapshot_cycles': snapshot_cycles,
        'ry_values': ry_values,
        'corr_sq_mean': corr_mean,
        'corr_sq_std': corr_std,
        'corr_sq_sem': corr_sem,
        'xavg_corr_mean': xavg_mean,
        'xavg_corr_sem': xavg_sem,
        'sample_count': sample_count,
    }


run_payloads = []
for record in SOURCE_RUNS:
    print(f"[compute] {record['case_id']}")
    run_payloads.append(compute_record_square_correlations(record))

write_json_atomic(OUTPUT_ROOT / 'validation_report.json', validation_report)
print(f'[validation] wrote {OUTPUT_ROOT / "validation_report.json"}')


[compute] N16x30_nsh1_perfect_correction
[spotcheck] N16x30_nsh1_perfect_correction max_abs_diff=8.882e-16
[compute] N16x30_nsh2_perfect_correction
[spotcheck] N16x30_nsh2_perfect_correction max_abs_diff=8.882e-16
[compute] N16x40_nsh1_perfect_correction
[spotcheck] N16x40_nsh1_perfect_correction max_abs_diff=8.882e-16
[compute] N16x40_nsh2_perfect_correction
[spotcheck] N16x40_nsh2_perfect_correction max_abs_diff=8.882e-16
[validation] wrote /home/abhuiyan/class_A_fermionic_adaptive_circuit/colab_small_system_testing/analysis_outputs/pure_state_square_correlations_cpu/validation_report.json


## 5. Exact DW Ground-State References

Compute exact equilibrium DW ground-state references for `N16x30` and `N16x40` using the same square-correlation observable and fit convention.


In [5]:
exact_references = {}
for ny in sorted({int(run['record']['Ny']) for run in run_payloads}):
    print(f'[exact] N16x{ny}')
    exact_references[ny] = exact_square_corr_reference(16, ny)
    fit = fit_corr_curve(exact_references[ny]['ry_values'], exact_references[ny]['xavg_corr'], ny)
    print(f"[exact] N16x{ny} xavg slope={fit['slope']:.4g}, R2={fit['r2']:.4f}, DWs={exact_references[ny]['domain_wall_x']}")


[exact] N16x30
[exact] N16x30 xavg slope=-2.095, R2=0.9989, DWs=(5, 11)
[exact] N16x40
[exact] N16x40 xavg slope=-2.089, R2=0.9990, DWs=(5, 11)


## 6. Build Tables And Fits

Flatten the dynamic correlation products into CSV-friendly tables and compute both x-averaged and fixed-`x` log-chord fit rows.


In [6]:
run_rows = []
corr_rows = []
xavg_rows = []
fit_rows = []

for payload in run_payloads:
    record = payload['record']
    nx, ny, nshell = int(record['Nx']), int(record['Ny']), int(record['nshell'])
    case_id = record['case_id']
    dw0, dw1 = record['domain_wall_x']
    ry_values = payload['ry_values']
    sample_count = int(payload['sample_count'])

    run_rows.append({
        'case_id': case_id,
        'Nx': nx,
        'Ny': ny,
        'nshell': nshell,
        'protocol': record['protocol'],
        'samples': sample_count,
        'cycles': int(record['cycles']),
        'snapshot_cycles': json.dumps(payload['snapshot_cycles'].astype(int).tolist()),
        'domain_wall_x': json.dumps([int(dw0), int(dw1)]),
        'source_shard_path': rel_to_bundle(record['shard_path']),
    })

    chord = chord_length(ry_values, ny)
    for t_idx, cycle in enumerate(payload['snapshot_cycles']):
        cycle = int(cycle)
        xavg = payload['xavg_corr_mean'][t_idx]
        xavg_sem = payload['xavg_corr_sem'][t_idx]
        for r_idx, ry in enumerate(ry_values):
            ry = int(ry)
            xavg_rows.append({
                'case_id': case_id,
                'source_kind': 'adaptive_circuit_snapshot',
                'Nx': nx,
                'Ny': ny,
                'nshell': nshell,
                'cycle': cycle,
                'ry': ry,
                'chord_length': float(chord[r_idx]),
                'xavg_corr_mean': float(xavg[r_idx]),
                'xavg_corr_sem': float(xavg_sem[r_idx]),
            })
            for x in range(nx):
                corr_rows.append({
                    'case_id': case_id,
                    'source_kind': 'adaptive_circuit_snapshot',
                    'Nx': nx,
                    'Ny': ny,
                    'nshell': nshell,
                    'cycle': cycle,
                    'x': int(x),
                    'ry': ry,
                    'chord_length': float(chord[r_idx]),
                    'corr_sq_mean': float(payload['corr_sq_mean'][t_idx, x, r_idx]),
                    'corr_sq_std': float(payload['corr_sq_std'][t_idx, x, r_idx]),
                    'corr_sq_sem': float(payload['corr_sq_sem'][t_idx, x, r_idx]),
                })

        if cycle == PRIMARY_CYCLE:
            xavg_fit = fit_corr_curve(ry_values, xavg, ny)
            fit_rows.append({
                'case_id': case_id,
                'source_kind': 'adaptive_circuit_snapshot',
                'observable': 'xavg_corr',
                'Nx': nx,
                'Ny': ny,
                'nshell': nshell,
                'cycle': cycle,
                'x': -1,
                'ry_fit_min': xavg_fit['ry_fit_min'],
                'ry_fit_max': xavg_fit['ry_fit_max'],
                'n_fit_points': xavg_fit['n_fit_points'],
                'slope': xavg_fit['slope'],
                'intercept': xavg_fit['intercept'],
                'r2': xavg_fit['r2'],
            })
            for x in range(nx):
                fixed_fit = fit_corr_curve(ry_values, payload['corr_sq_mean'][t_idx, x], ny)
                fit_rows.append({
                    'case_id': case_id,
                    'source_kind': 'adaptive_circuit_snapshot',
                    'observable': 'fixed_x_corr',
                    'Nx': nx,
                    'Ny': ny,
                    'nshell': nshell,
                    'cycle': cycle,
                    'x': int(x),
                    'ry_fit_min': fixed_fit['ry_fit_min'],
                    'ry_fit_max': fixed_fit['ry_fit_max'],
                    'n_fit_points': fixed_fit['n_fit_points'],
                    'slope': fixed_fit['slope'],
                    'intercept': fixed_fit['intercept'],
                    'r2': fixed_fit['r2'],
                })

for ny, ref in exact_references.items():
    case_id = f'exact_dw_gs_N16x{ny}'
    ry_values = ref['ry_values']
    chord = chord_length(ry_values, ny)
    xavg = ref['xavg_corr']
    xavg_fit = fit_corr_curve(ry_values, xavg, ny)
    fit_rows.append({
        'case_id': case_id,
        'source_kind': 'exact_dw_gs',
        'observable': 'xavg_corr',
        'Nx': int(ref['Nx']),
        'Ny': int(ref['Ny']),
        'nshell': -1,
        'cycle': -1,
        'x': -1,
        'ry_fit_min': xavg_fit['ry_fit_min'],
        'ry_fit_max': xavg_fit['ry_fit_max'],
        'n_fit_points': xavg_fit['n_fit_points'],
        'slope': xavg_fit['slope'],
        'intercept': xavg_fit['intercept'],
        'r2': xavg_fit['r2'],
    })
    for r_idx, ry in enumerate(ry_values):
        xavg_rows.append({
            'case_id': case_id,
            'source_kind': 'exact_dw_gs',
            'Nx': int(ref['Nx']),
            'Ny': int(ref['Ny']),
            'nshell': -1,
            'cycle': -1,
            'ry': int(ry),
            'chord_length': float(chord[r_idx]),
            'xavg_corr_mean': float(xavg[r_idx]),
            'xavg_corr_sem': 0.0,
        })
        for x in range(int(ref['Nx'])):
            corr_rows.append({
                'case_id': case_id,
                'source_kind': 'exact_dw_gs',
                'Nx': int(ref['Nx']),
                'Ny': int(ref['Ny']),
                'nshell': -1,
                'cycle': -1,
                'x': int(x),
                'ry': int(ry),
                'chord_length': float(chord[r_idx]),
                'corr_sq_mean': float(ref['corr_sq'][x, r_idx]),
                'corr_sq_std': 0.0,
                'corr_sq_sem': 0.0,
            })
    for x in range(int(ref['Nx'])):
        fixed_fit = fit_corr_curve(ry_values, ref['corr_sq'][x], ny)
        fit_rows.append({
            'case_id': case_id,
            'source_kind': 'exact_dw_gs',
            'observable': 'fixed_x_corr',
            'Nx': int(ref['Nx']),
            'Ny': int(ref['Ny']),
            'nshell': -1,
            'cycle': -1,
            'x': int(x),
            'ry_fit_min': fixed_fit['ry_fit_min'],
            'ry_fit_max': fixed_fit['ry_fit_max'],
            'n_fit_points': fixed_fit['n_fit_points'],
            'slope': fixed_fit['slope'],
            'intercept': fixed_fit['intercept'],
            'r2': fixed_fit['r2'],
        })

run_index = pd.DataFrame(run_rows).sort_values(['Ny', 'nshell']).reset_index(drop=True)
corr_curves = pd.DataFrame(corr_rows).sort_values(['source_kind', 'Ny', 'nshell', 'cycle', 'x', 'ry']).reset_index(drop=True)
xavg_corr_curves = pd.DataFrame(xavg_rows).sort_values(['source_kind', 'Ny', 'nshell', 'cycle', 'ry']).reset_index(drop=True)
log_chord_fit_rows = pd.DataFrame(fit_rows).sort_values(['source_kind', 'observable', 'Ny', 'nshell', 'cycle', 'x']).reset_index(drop=True)
log_chord_fit_summary = (
    log_chord_fit_rows
    .groupby(['source_kind', 'observable', 'Nx', 'Ny', 'nshell', 'cycle'], dropna=False)
    .agg(
        mean_slope=('slope', 'mean'),
        min_slope=('slope', 'min'),
        max_slope=('slope', 'max'),
        mean_r2=('r2', 'mean'),
        fit_count=('slope', 'count'),
    )
    .reset_index()
    .sort_values(['source_kind', 'observable', 'Ny', 'nshell', 'cycle'])
)

run_index.to_csv(OUTPUT_ROOT / 'run_index.csv', index=False)
corr_curves.to_csv(OUTPUT_ROOT / 'corr_curves.csv', index=False)
xavg_corr_curves.to_csv(OUTPUT_ROOT / 'xavg_corr_curves.csv', index=False)
log_chord_fit_rows.to_csv(OUTPUT_ROOT / 'log_chord_fit_rows.csv', index=False)
log_chord_fit_summary.to_csv(OUTPUT_ROOT / 'log_chord_fit_summary.csv', index=False)

print('[tables] wrote CSV outputs')
display(Markdown('### Run index'))
display(run_index)
display(Markdown(f'### Cycle {PRIMARY_CYCLE} x-averaged log-chord fits'))
display(log_chord_fit_rows[(log_chord_fit_rows['observable'] == 'xavg_corr')].reset_index(drop=True))


[tables] wrote CSV outputs


### Run index

,case_id,Nx,Ny,nshell,protocol,samples,cycles,snapshot_cycles,domain_wall_x,source_shard_path
0,N16x30_nsh1_perfect_correction,16,30,1,perfect_correction,10,50,"[5, 10, 20, 50]","[5, 11]",gpu_data/pure_state_covariance_snapshots/runs/...
1,N16x30_nsh2_perfect_correction,16,30,2,perfect_correction,10,50,"[5, 10, 20, 50]","[5, 11]",gpu_data/pure_state_covariance_snapshots/runs/...
2,N16x40_nsh1_perfect_correction,16,40,1,perfect_correction,10,50,"[5, 10, 20, 50]","[5, 11]",gpu_data/pure_state_covariance_snapshots/runs/...
3,N16x40_nsh2_perfect_correction,16,40,2,perfect_correction,10,50,"[5, 10, 20, 50]","[5, 11]",gpu_data/pure_state_covariance_snapshots/runs/...


### x-averaged log-chord fits

,case_id,source_kind,observable,Nx,Ny,nshell,cycle,x,ry_fit_min,ry_fit_max,n_fit_points,slope,intercept,r2
0,N16x30_nsh1_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,1,5,-1,3,10,8,-1.747610,-7.141588,0.937862
1,N16x30_nsh1_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,1,10,-1,3,10,8,-2.470605,-5.889008,0.999791
2,N16x30_nsh1_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,1,20,-1,3,10,8,-2.469602,-5.809616,0.998727
3,N16x30_nsh1_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,1,50,-1,3,10,8,-2.298144,-6.010194,0.999813
4,N16x30_nsh2_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,2,5,-1,3,10,8,-1.776191,-6.988423,0.933140
5,N16x30_nsh2_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,2,10,-1,3,10,8,-2.434328,-5.861887,0.998566
6,N16x30_nsh2_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,2,20,-1,3,10,8,-2.370629,-5.767582,0.998867
7,N16x30_nsh2_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,30,2,50,-1,3,10,8,-2.355089,-5.827236,0.999813
8,N16x40_nsh1_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,40,1,5,-1,3,10,8,-1.450840,-7.679126,0.895907
9,N16x40_nsh1_perfect_correction,adaptive_circuit_snapshot,xavg_corr,16,40,1,10,-1,3,10,8,-2.396241,-6.087930,0.997048


## 7. Save Reusable Padded Arrays

Write compact arrays for later plotting without reopening the original covariance shards. `Ny=30` entries are padded with `NaN` up to the `Ny=40` `ry` length.


In [7]:
case_ids = [payload['record']['case_id'] for payload in run_payloads]
max_ry_count = max(int(payload['ry_values'].size) for payload in run_payloads)
max_x = max(int(payload['record']['Nx']) for payload in run_payloads)
time_count = len(EXPECTED_SNAPSHOT_CYCLES)

corr_sq_mean_padded = np.full((len(run_payloads), time_count, max_x, max_ry_count), np.nan, dtype=np.float64)
corr_sq_sem_padded = np.full_like(corr_sq_mean_padded, np.nan)
xavg_corr_mean_padded = np.full((len(run_payloads), time_count, max_ry_count), np.nan, dtype=np.float64)
xavg_corr_sem_padded = np.full_like(xavg_corr_mean_padded, np.nan)
ry_values_padded = np.full((len(run_payloads), max_ry_count), -1, dtype=np.int64)

for run_idx, payload in enumerate(run_payloads):
    nx = int(payload['record']['Nx'])
    r_count = int(payload['ry_values'].size)
    corr_sq_mean_padded[run_idx, :, :nx, :r_count] = payload['corr_sq_mean']
    corr_sq_sem_padded[run_idx, :, :nx, :r_count] = payload['corr_sq_sem']
    xavg_corr_mean_padded[run_idx, :, :r_count] = payload['xavg_corr_mean']
    xavg_corr_sem_padded[run_idx, :, :r_count] = payload['xavg_corr_sem']
    ry_values_padded[run_idx, :r_count] = payload['ry_values']

exact_Ny_values = np.asarray(sorted(exact_references), dtype=np.int64)
exact_corr_sq_padded = np.full((len(exact_Ny_values), max_x, max_ry_count), np.nan, dtype=np.float64)
exact_xavg_corr_padded = np.full((len(exact_Ny_values), max_ry_count), np.nan, dtype=np.float64)
exact_ry_values_padded = np.full((len(exact_Ny_values), max_ry_count), -1, dtype=np.int64)
for idx, ny in enumerate(exact_Ny_values):
    ref = exact_references[int(ny)]
    r_count = int(ref['ry_values'].size)
    exact_corr_sq_padded[idx, :int(ref['Nx']), :r_count] = ref['corr_sq']
    exact_xavg_corr_padded[idx, :r_count] = ref['xavg_corr']
    exact_ry_values_padded[idx, :r_count] = ref['ry_values']

if corr_sq_mean_padded.shape != (4, 4, 16, 21):
    raise ValueError(f'Unexpected padded corr shape {corr_sq_mean_padded.shape}')
if xavg_corr_mean_padded.shape != (4, 4, 21):
    raise ValueError(f'Unexpected padded xavg shape {xavg_corr_mean_padded.shape}')

save_npz_atomic(
    OUTPUT_ROOT / 'derived_arrays.npz',
    case_id=np.asarray(case_ids),
    snapshot_cycles=EXPECTED_SNAPSHOT_CYCLES,
    Nx_values=np.asarray([int(p['record']['Nx']) for p in run_payloads], dtype=np.int64),
    Ny_values=np.asarray([int(p['record']['Ny']) for p in run_payloads], dtype=np.int64),
    nshell_values=np.asarray([int(p['record']['nshell']) for p in run_payloads], dtype=np.int64),
    domain_wall_x=np.asarray([p['record']['domain_wall_x'] for p in run_payloads], dtype=np.int64),
    ry_values_padded=ry_values_padded,
    corr_sq_mean_padded=corr_sq_mean_padded,
    corr_sq_sem_padded=corr_sq_sem_padded,
    xavg_corr_mean_padded=xavg_corr_mean_padded,
    xavg_corr_sem_padded=xavg_corr_sem_padded,
    exact_Ny_values=exact_Ny_values,
    exact_ry_values_padded=exact_ry_values_padded,
    exact_corr_sq_padded=exact_corr_sq_padded,
    exact_xavg_corr_padded=exact_xavg_corr_padded,
    config_json=np.asarray(json.dumps({
        'analysis_name': ANALYSIS_NAME,
        'source_manifest': rel_to_bundle(SOURCE_MANIFEST_PATH),
        'formula': 'C=(G+I)/2; C_G(x,ry)=(1/(2*Ny))*sum_y sum_mu,nu |C[x,y,mu;x,y+ry,nu]|^2',
        'fit_window_ry': [CORR_FIT_RY_MIN, CORR_FIT_RY_MAX],
        'primary_cycle': PRIMARY_CYCLE,
    }, sort_keys=True)),
)
print('[arrays] wrote derived_arrays.npz')


[arrays] wrote derived_arrays.npz


## 8. Correlation Profiles Vs Cycle

This is the notebook's time-resolved section. Each code cell focuses on one system size and plots fixed-`x` square-correlation profiles across the saved cycles `[5, 10, 20, 50]`, with separate rows for `nshell=1` and `nshell=2`.


### N16x30 Profiles

In [ ]:
ny_target = 30
payloads = [payload for payload in run_payloads if int(payload['record']['Ny']) == ny_target]
payloads = sorted(payloads, key=lambda item: int(item['record']['nshell']))
fig, axes = plt.subplots(len(payloads), len(EXPECTED_SNAPSHOT_CYCLES), figsize=(16.0, 3.6 * len(payloads)), constrained_layout=True, sharex=False, sharey=False)
axes = np.atleast_2d(axes)

for row_idx, payload in enumerate(payloads):
    record = payload['record']
    nx, ny = int(record['Nx']), int(record['Ny'])
    xs = selected_x_values(nx)
    ry = payload['ry_values']
    valid = (ry > 0) & (ry < ny / 2)
    for col_idx, cycle in enumerate(EXPECTED_SNAPSHOT_CYCLES):
        ax = axes[row_idx, col_idx]
        t_idx = int(np.where(EXPECTED_SNAPSHOT_CYCLES == cycle)[0][0])
        for x in xs:
            vals = payload['corr_sq_mean'][t_idx, x]
            good = valid & (vals > 0) & np.isfinite(vals)
            if np.any(good):
                ax.loglog(ry[good], vals[good], marker='o', ms=2.3, lw=0.8, label=f'x={x}')
        for xdw in record['domain_wall_x']:
            ax.text(0.03, 0.05 + 0.08 * list(record['domain_wall_x']).index(xdw), f'DW x={xdw}', transform=ax.transAxes, fontsize=7)
        ax.set_title(f"nshell={record['nshell']} | cycle {int(cycle)}", fontsize=9)
        ax.set_xlabel('ry')
        ax.set_ylabel(r'$C_G(x, r_y)$')
        ax.legend(frameon=True, fontsize=6, ncol=2)

fig.suptitle('N16x30 fixed-x square-correlation profiles vs cycle', y=1.002)
fig_path = FIG_ROOT / 'fixed_x_profiles_N16x30_by_cycle.png'
fig.savefig(fig_path)
plt.close(fig)
print(f'[figures] wrote {fig_path}')
show_saved_figure(fig_path, 'N16x30 fixed-x square-correlation profiles vs cycle')


### N16x40 Profiles

In [ ]:
ny_target = 40
payloads = [payload for payload in run_payloads if int(payload['record']['Ny']) == ny_target]
payloads = sorted(payloads, key=lambda item: int(item['record']['nshell']))
fig, axes = plt.subplots(len(payloads), len(EXPECTED_SNAPSHOT_CYCLES), figsize=(16.0, 3.6 * len(payloads)), constrained_layout=True, sharex=False, sharey=False)
axes = np.atleast_2d(axes)

for row_idx, payload in enumerate(payloads):
    record = payload['record']
    nx, ny = int(record['Nx']), int(record['Ny'])
    xs = selected_x_values(nx)
    ry = payload['ry_values']
    valid = (ry > 0) & (ry < ny / 2)
    for col_idx, cycle in enumerate(EXPECTED_SNAPSHOT_CYCLES):
        ax = axes[row_idx, col_idx]
        t_idx = int(np.where(EXPECTED_SNAPSHOT_CYCLES == cycle)[0][0])
        for x in xs:
            vals = payload['corr_sq_mean'][t_idx, x]
            good = valid & (vals > 0) & np.isfinite(vals)
            if np.any(good):
                ax.loglog(ry[good], vals[good], marker='o', ms=2.3, lw=0.8, label=f'x={x}')
        for xdw in record['domain_wall_x']:
            ax.text(0.03, 0.05 + 0.08 * list(record['domain_wall_x']).index(xdw), f'DW x={xdw}', transform=ax.transAxes, fontsize=7)
        ax.set_title(f"nshell={record['nshell']} | cycle {int(cycle)}", fontsize=9)
        ax.set_xlabel('ry')
        ax.set_ylabel(r'$C_G(x, r_y)$')
        ax.legend(frameon=True, fontsize=6, ncol=2)

fig.suptitle('N16x40 fixed-x square-correlation profiles vs cycle', y=1.002)
fig_path = FIG_ROOT / 'fixed_x_profiles_N16x40_by_cycle.png'
fig.savefig(fig_path)
plt.close(fig)
print(f'[figures] wrote {fig_path}')
show_saved_figure(fig_path, 'N16x40 fixed-x square-correlation profiles vs cycle')


## 9. Cycle-50 x-Averaged Log-Chord Fits

All remaining characterization cells are restricted to the final saved snapshot, `cycle=50`. This cell compares the adaptive-circuit `x`-averaged correlator against the exact DW ground-state reference for each run.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5), constrained_layout=True, sharey=True)
axes = axes.ravel()
for ax, payload in zip(axes, run_payloads):
    record = payload['record']
    ny = int(record['Ny'])
    ry = payload['ry_values']
    t_idx = int(np.where(EXPECTED_SNAPSHOT_CYCLES == PRIMARY_CYCLE)[0][0])
    xavg = payload['xavg_corr_mean'][t_idx]
    chord = chord_length(ry, ny)
    valid = (ry > 0) & (ry < ny / 2) & (chord > 0) & (xavg > 0)
    fit_window = valid & (ry >= CORR_FIT_RY_MIN) & (ry <= CORR_FIT_RY_MAX)
    fit = fit_corr_curve(ry, xavg, ny)
    if np.any(fit_window):
        ax.axvspan(float(np.min(np.log(chord[fit_window]))), float(np.max(np.log(chord[fit_window]))), color='0.6', alpha=0.14, zorder=0, label='fit window')
    ax.plot(np.log(chord[valid]), np.log(xavg[valid]), 'o', ms=3, label=f'cycle {PRIMARY_CYCLE} data')
    lx = np.linspace(float(np.min(np.log(chord[valid]))), float(np.max(np.log(chord[valid]))), 200)
    ax.plot(lx, fit['slope'] * lx + fit['intercept'], lw=1.4, label=f"cycle {PRIMARY_CYCLE} m={fit['slope']:.3g}")

    ref = exact_references[ny]
    ref_xavg = ref['xavg_corr']
    ref_chord = chord_length(ref['ry_values'], ny)
    ref_valid = (ref['ry_values'] > 0) & (ref['ry_values'] < ny / 2) & (ref_chord > 0) & (ref_xavg > 0)
    ref_fit = fit_corr_curve(ref['ry_values'], ref_xavg, ny)
    ax.plot(np.log(ref_chord[ref_valid]), np.log(ref_xavg[ref_valid]), 's', ms=3, color='black', label='exact DW GS')
    lx_ref = np.linspace(float(np.min(np.log(ref_chord[ref_valid]))), float(np.max(np.log(ref_chord[ref_valid]))), 200)
    ax.plot(lx_ref, ref_fit['slope'] * lx_ref + ref_fit['intercept'], lw=1.2, color='tab:red', label=f"exact m={ref_fit['slope']:.3g}")
    ax.set_title(record['case_id'])
    ax.set_xlabel('log chord(ry)')
    ax.set_ylabel(r'log $\langle C_G(r_y)\rangle_x$')
    ax.legend(frameon=True, fontsize=7)
fig.suptitle(f'Cycle {PRIMARY_CYCLE} x-averaged square correlator vs exact DW GS', y=1.002)
fig_path = FIG_ROOT / 'xavg_log_chord_fits_cycle50.png'
fig.savefig(fig_path)
plt.close(fig)
print(f'[figures] wrote {fig_path}')
show_saved_figure(fig_path, f'Cycle {PRIMARY_CYCLE} x-averaged log-chord fits vs exact DW GS')


## 10. Cycle-50 Fixed-`x` Slope Diagnostics

Fit `log C_G(x, ry)` versus `log chord(ry)` at each fixed `x`, using only `cycle=50`, and compare the fitted slopes to the exact DW ground-state reference.


In [ ]:
fixed_fit_df = log_chord_fit_rows[
    (log_chord_fit_rows['observable'] == 'fixed_x_corr')
    & (log_chord_fit_rows['source_kind'] == 'adaptive_circuit_snapshot')
    & (log_chord_fit_rows['cycle'] == PRIMARY_CYCLE)
].copy()

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.8), constrained_layout=True, sharex=True, sharey=True)
axes = axes.ravel()
for ax, payload in zip(axes, run_payloads):
    record = payload['record']
    ny = int(record['Ny'])
    sub = fixed_fit_df[fixed_fit_df['case_id'] == record['case_id']].sort_values('x')
    exact_sub = log_chord_fit_rows[
        (log_chord_fit_rows['case_id'] == f'exact_dw_gs_N16x{ny}')
        & (log_chord_fit_rows['observable'] == 'fixed_x_corr')
    ].sort_values('x')
    ax.plot(sub['x'], sub['slope'], marker='o', lw=1.3, label=f'cycle {PRIMARY_CYCLE}')
    ax.plot(exact_sub['x'], exact_sub['slope'], marker='s', lw=1.1, color='black', label='exact DW GS')
    ax.axhline(-2.0, color='0.4', ls=':', lw=1.0, label='slope -2')
    for xdw in record['domain_wall_x']:
        ax.axvline(float(xdw), color='tab:red', ls='--', lw=1.0, alpha=0.8)
    ax.set_title(record['case_id'])
    ax.set_xlabel('x')
    ax.set_ylabel('fixed-x fitted slope')
    ax.legend(frameon=True, fontsize=7)
fig.suptitle(f'Fixed-x log-chord slopes at cycle {PRIMARY_CYCLE} vs exact references', y=1.002)
fig_path = FIG_ROOT / 'fixed_x_slope_cycle50_vs_exact.png'
fig.savefig(fig_path)
plt.close(fig)
print(f'[figures] wrote {fig_path}')
show_saved_figure(fig_path, f'Fixed-x slopes at cycle {PRIMARY_CYCLE} vs exact references')


## 11. Output Manifest

Verify that the expected products exist and record the analysis metadata needed to reproduce the notebook outputs.


In [ ]:
expected_outputs = {
    'analysis_manifest.json',
    'run_index.csv',
    'corr_curves.csv',
    'xavg_corr_curves.csv',
    'log_chord_fit_rows.csv',
    'log_chord_fit_summary.csv',
    'validation_report.json',
    'derived_arrays.npz',
}
expected_figures = {
    'fixed_x_profiles_N16x30_by_cycle.png',
    'fixed_x_profiles_N16x40_by_cycle.png',
    'xavg_log_chord_fits_cycle50.png',
    'fixed_x_slope_cycle50_vs_exact.png',
}

analysis_manifest = {
    'analysis_name': ANALYSIS_NAME,
    'source_campaign': SOURCE_CAMPAIGN,
    'source_manifest': rel_to_bundle(SOURCE_MANIFEST_PATH),
    'canonical_source_entry_point': CANONICAL_SOURCE_ENTRY_POINT,
    'output_root': rel_to_bundle(OUTPUT_ROOT),
    'figures_root': rel_to_bundle(FIG_ROOT),
    'formula_convention': 'C=(G+I)/2; C_G(x,ry)=(1/(2*Ny))*sum_y sum_mu,nu |C[x,y,mu;x,y+ry,nu]|^2 with periodic y',
    'snapshot_cycles': EXPECTED_SNAPSHOT_CYCLES.astype(int).tolist(),
    'profile_section_cycles': EXPECTED_SNAPSHOT_CYCLES.astype(int).tolist(),
    'characterization_cycle': int(PRIMARY_CYCLE),
    'fit_window_ry': [int(CORR_FIT_RY_MIN), int(CORR_FIT_RY_MAX)],
    'exact_references': [f'exact_dw_gs_N16x{int(ny)}' for ny in exact_Ny_values],
    'run_count': int(len(run_index)),
    'corr_curve_rows': int(len(corr_curves)),
    'xavg_corr_curve_rows': int(len(xavg_corr_curves)),
    'fit_rows': int(len(log_chord_fit_rows)),
    'outputs': sorted(expected_outputs),
    'figures': sorted(expected_figures),
    'notes': [
        'This local CPU notebook does not run adaptive-circuit dynamics.',
        'The profile section is the only time-resolved characterization and is split into one cell per system size.',
        'Log-chord fits and fixed-x slope diagnostics are restricted to cycle 50 plus exact references.',
        'SEM for x-averaged correlations is propagated from fixed-x SEM values assuming no cross-x covariance.',
    ],
}
write_json_atomic(OUTPUT_ROOT / 'analysis_manifest.json', analysis_manifest)

output_files = sorted(p.name for p in OUTPUT_ROOT.iterdir() if p.is_file())
figure_files = sorted(p.name for p in FIG_ROOT.iterdir() if p.is_file())
missing_outputs = sorted(expected_outputs - set(output_files))
missing_figures = sorted(expected_figures - set(figure_files))
if missing_outputs:
    raise RuntimeError(f'Missing expected outputs: {missing_outputs}')
if missing_figures:
    raise RuntimeError(f'Missing expected figures: {missing_figures}')

print(f'[done] outputs written to {OUTPUT_ROOT}')
print(f'[done] figures written to {FIG_ROOT}')
display(Markdown('### Output manifest'))
display(pd.DataFrame({'outputs': sorted(expected_outputs) + [f'figures/{name}' for name in sorted(expected_figures)]}))
display(Markdown(f'Analysis output folder: `{rel_to_bundle(OUTPUT_ROOT)}`'))
